# 0415 · LightGBM (ensamble), versión 2

> **Versión 2 (05-11-2026, evaluación del director).** Lee `V4/MODELADO/` (0411 v2, cinco bloques de abril a agosto) y comprueba la huella de la base antes de entrenar (A6). **C3:** en regresión la búsqueda reparte las configuraciones por igual entre las funciones de pérdida (L2, L1 y Huber) y, además del mejor modelo, valida el mejor de cada pérdida (`LightGBM_L2`, `LightGBM_L1`, `LightGBM_Huber`), para compararlo en 0416 con la referencia de su mismo tipo (media para L2, mediana para L1).

**Qué hace.** Entrena y valida solo LightGBM, en las dos tareas:

| Tarea | Qué predice | Métrica para elegir |
| --- | --- | --- |
| `clf` | Probabilidad de incumplir el SLA | AP (mayor es mejor) |
| `reg` | Horas de última milla | MAE en horas (menor es mejor) |

**LightGBM** también es *boosting* de árboles, pero los hace crecer por hojas y agrupa los valores en histogramas, así que suele ser el más rápido con muchas guías. En regresión la búsqueda también elige la función de pérdida (`objetivo_reg`).

**Pasos.** 1) carga lo que dejó 0411; 2) búsqueda aleatoria de hiperparámetros con validación de origen móvil; 3) validación con la mejor configuración en los tres bloques (abril, mayo, junio); 4) vista rápida de sus métricas; 5) guarda predicciones, importancias y parámetros para 0416.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

**Tiempo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 bloques × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 100.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 94.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 158.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 13.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 131.4 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")   # E1: sin avisos que impriman rutas

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, muestrear_configs, puntaje_config, mejores, params_de,
                                       importancia_pliegue, tabla_metricas, verificar_huella,
                                       PERDIDA_CORTA, mejores_por_perdida)

In [0]:
MODELO = "LightGBM"
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_11/V4/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la validación)
EJECUTAR_BUSQUEDA  = True      # False: usa los parámetros que ya guardó este cuaderno
N_PERMUTACION      = 10_000    # filas de validación para la importancia por permutación en cada bloque
EVALUAR_POR_PERDIDA = True     # C3: valida también el mejor de cada función de pérdida (solo regresión, variables internas)

In [0]:
# Hiperparámetros de base (si no hay búsqueda) y espacio de búsqueda registrado (realimentación P4)
PARAMS_BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8)
ESPACIO = dict(n_estimators=[200, 400, 800], learning_rate=[0.02, 0.05, 0.1], num_leaves=[31, 63, 127, 255],
               min_child_samples=[20, 50, 100, 200], subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
               reg_lambda=[0.0, 1.0, 5.0], objetivo_reg=["regression", "regression_l1", "huber"])

## 1. Cargar la preparación de 0411

Base de modelado, historia sin futuro de cada bloque y configuración común (periodos, semilla, variables). Los pliegues se vuelven a armar con la misma función del asesor.

In [0]:
print("Huella de la base comprobada:", verificar_huella(CARPETA_MODELADO)[:16], "...")   # A6
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

Huella de la base comprobada: dc7400e56c1f74c7 ...
1,553,980 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-11-07T14:07:13.019713


## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Se prueban `N_CONFIGS_BUSQUEDA` configuraciones de `ESPACIO`. Cada una se entrena en cada bloque con una submuestra de hasta `MAX_FILAS_BUSQUEDA` filas y se evalúa en el bloque completo. Se elige la de mejor promedio entre bloques. Si la base trae la versión «internas+clima_reajustada», se busca también para ella.

In [0]:
def buscar(version, numericas):
    """Prueba todas las configuraciones de una versión de variables; una fila por configuración."""
    filas = []
    for tarea, y_col, _ in TAREAS:
        for k, cfg in enumerate(muestrear_configs(ESPACIO, N_CONFIGS_BUSQUEDA, SEMILLA + len(MODELO))):
            if tarea == "clf":
                cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}   # solo aplica a regresión
            else:   # C3: las configuraciones se reparten por igual entre las funciones de pérdida
                cfg["objetivo_reg"] = ESPACIO["objetivo_reg"][k % len(ESPACIO["objetivo_reg"])]
            vals = puntaje_config(tarea, MODELO, cfg, k, PLIEGUES[tarea], HIST, df, y_col,
                                  CATEGORICAS, numericas, MAX_FILAS_BUSQUEDA, SEMILLA)
            filas.append({"version": version, "tarea": tarea, "modelo": MODELO, "config": k,
                          "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                          "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
            print(f"{version} {tarea} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

In [0]:
RUTA_PARAMS = f"{CARPETA_MODELADO}parametros_{MODELO}.json"
t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    versiones_busqueda = ["internas"] + [v for v in VERSIONES if v == "internas+clima_reajustada"]
    tabla_busqueda = pd.concat([buscar(v, VERSIONES[v]) for v in versiones_busqueda], ignore_index=True)
    tabla_busqueda.to_pickle(f"{CARPETA_MODELADO}busqueda_{MODELO}.pkl")       # exacta, para 0416
    tabla_busqueda.to_csv(f"{CARPETA_MODELADO}busqueda_{MODELO}.csv", index=False)
    MEJORES = mejores(tabla_busqueda, {MODELO: ESPACIO})
    MEJORES_POR_PERDIDA = mejores_por_perdida(tabla_busqueda, ESPACIO, MODELO)
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
    MEJORES_POR_PERDIDA = json.load(open(RUTA_PARAMS)).get("mejores_por_perdida", {})
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)
print("Mejor MAE de la búsqueda por función de pérdida (C3):")
display(pd.DataFrame({k: {"MAE_h": v["promedio_MAE_h"], "configs": v["n_configs"]} for k, v in MEJORES_POR_PERDIDA.items()}).T)

internas clf config 0: 0.2774
internas clf config 1: 0.3029
internas clf config 2: 0.2835
internas clf config 3: 0.3077
internas clf config 4: 0.3107
internas clf config 5: 0.2827
internas clf config 6: 0.2722
internas clf config 7: 0.2877
internas clf config 8: 0.2952
internas clf config 9: 0.3024
internas clf config 10: 0.3087
internas clf config 11: 0.2780
internas clf config 12: 0.2890
internas clf config 13: 0.2928
internas clf config 14: 0.3100
internas clf config 15: 0.2988
internas clf config 16: 0.2794
internas clf config 17: 0.2953
internas clf config 18: 0.3069
internas clf config 19: 0.3161
internas reg config 0: 8.1668
internas reg config 1: 6.5396
internas reg config 2: 6.7217
internas reg config 3: 7.2766
internas reg config 4: 6.5546
internas reg config 5: 6.7182
internas reg config 6: 7.5653
internas reg config 7: 6.5986
internas reg config 8: 6.6446
internas reg config 9: 7.5036
internas reg config 10: 6.5757
internas reg config 11: 6.5772
internas reg config 12: 7.61

n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda,objetivo_reg
200.0,0.02,255.0,100.0,0.8,0.6,5.0,null
800.0,0.02,63.0,100.0,1.0,0.6,5.0,regression_l1


Mejor MAE de la búsqueda por función de pérdida (C3):


MAE_h,configs
6.5771539879312515,6.0
7.276590386279738,7.0
6.539616395557419,7.0


## 3. Validación de origen móvil con la mejor configuración

En cada versión de variables, tarea y bloque: se entrena con todo lo anterior al bloque, se predice el bloque y se guardan las predicciones. Con las variables internas se mide además la importancia por permutación (insumo de 0420).

In [0]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        params = params_de(MEJORES, {MODELO: PARAMS_BASE}, version, tarea, MODELO)
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred.append(marco_predicciones(version, tarea, ini, Dva, yva).assign(modelo=MODELO, pred=predecir(m, Xva, tarea)))
            if version == "internas":
                etiquetas = {"tarea": tarea, "modelo": MODELO, "bloque": ini.date()}
                imp.append(importancia_pliegue(m, Xva, yva, tarea, N_PERMUTACION, SEMILLA, etiquetas))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred, imp = pd.concat(pred, ignore_index=True), pd.concat(imp, ignore_index=True)

internas · clf · 2026-04-01 listo (58 s)
internas · clf · 2026-05-01 listo (124 s)
internas · clf · 2026-06-01 listo (197 s)
internas · clf · 2026-07-01 listo (280 s)
internas · clf · 2026-08-01 listo (367 s)
internas · reg · 2026-04-01 listo (476 s)
internas · reg · 2026-05-01 listo (592 s)
internas · reg · 2026-06-01 listo (719 s)
internas · reg · 2026-07-01 listo (856 s)
internas · reg · 2026-08-01 listo (1000 s)
internas+clima · clf · 2026-04-01 listo (1047 s)
internas+clima · clf · 2026-05-01 listo (1104 s)
internas+clima · clf · 2026-06-01 listo (1185 s)
internas+clima · clf · 2026-07-01 listo (1257 s)
internas+clima · clf · 2026-08-01 listo (1336 s)
internas+clima · reg · 2026-04-01 listo (1416 s)
internas+clima · reg · 2026-05-01 listo (1516 s)
internas+clima · reg · 2026-06-01 listo (1636 s)
internas+clima · reg · 2026-07-01 listo (1774 s)
internas+clima · reg · 2026-08-01 listo (1924 s)


### 3.1 Mejor modelo de cada función de pérdida (C3)

Solo regresión y variables internas. Cada uno se entrena con su mejor configuración de la búsqueda y queda en `pred_val_{MODELO}_por_perdida.parquet` para 0416.

In [0]:
pred_perdida = []
if EVALUAR_POR_PERDIDA and MEJORES_POR_PERDIDA:
    numericas, (tarea, y_col, _) = VERSIONES["internas"], [t for t in TAREAS if t[0] == "reg"][0]
    t_ini = time.time()
    for obj, info in MEJORES_POR_PERDIDA.items():
        params = params_de({f"internas|reg|{MODELO}": info["params"]}, {MODELO: PARAMS_BASE}, "internas", "reg", MODELO)
        etiqueta = f"{MODELO}_{PERDIDA_CORTA.get(obj, obj)}"
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred_perdida.append(marco_predicciones("internas", tarea, ini, Dva, yva).assign(modelo=etiqueta, pred=predecir(m, Xva, tarea)))
        print(f"{etiqueta} listo ({time.time() - t_ini:.0f} s)")
    pred_perdida = pd.concat(pred_perdida, ignore_index=True)
    display(tabla_metricas(pred_perdida, ["version", "tarea", "modelo"]).round(4))

LightGBM_Huber listo (784 s)
LightGBM_L2 listo (946 s)
LightGBM_L1 listo (1411 s)


version,tarea,modelo,n,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,reg,LightGBM_Huber,958475,6.5986,32.2377,7.1344,44.1563,59.7885,0.97
internas,reg,LightGBM_L1,958475,6.5619,32.1143,7.0498,43.6899,60.2393,0.97
internas,reg,LightGBM_L2,958475,7.3549,31.7638,9.5369,40.7901,98.9161,0.97


## 4. Vista rápida de LightGBM

Solo para revisar este modelo. La comparación con los demás se hace en 0416.

In [0]:
display(tabla_metricas(pred, ["version", "tarea", "modelo"]).round(4))
display(tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,LightGBM,966041,0.0765,0.2955,0.6975,0.0635,0.2533,null,null,null,null,null,null
internas,reg,LightGBM,958475,null,null,null,null,null,6.5619,32.1143,7.0498,43.6899,60.2393,0.97
internas+clima,clf,LightGBM,966041,0.0765,0.3062,0.6978,0.0626,0.25,null,null,null,null,null,null
internas+clima,reg,LightGBM,958475,null,null,null,null,null,6.5934,32.1073,7.0472,43.6642,61.8398,0.97


version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,LightGBM,2026-04-01,193214,0.0711,0.3536,0.6885,0.0561,0.2508,null,null,null,null,null,null
internas,clf,LightGBM,2026-05-01,204833,0.0857,0.3416,0.6927,0.0698,0.2898,null,null,null,null,null,null
internas,clf,LightGBM,2026-06-01,185344,0.082,0.3295,0.742,0.0651,0.2425,null,null,null,null,null,null
internas,clf,LightGBM,2026-07-01,246075,0.0715,0.2827,0.704,0.0589,0.2292,null,null,null,null,null,null
internas,clf,LightGBM,2026-08-01,136575,0.0717,0.2388,0.7084,0.0703,0.2602,null,null,null,null,null,null
internas,reg,LightGBM,2026-04-01,191924,null,null,null,null,null,6.1412,37.4707,6.6449,39.6619,60.2633,0.9666
internas,reg,LightGBM,2026-05-01,202331,null,null,null,null,null,6.7395,34.7704,7.754,45.3003,56.434,0.9647
internas,reg,LightGBM,2026-06-01,183841,null,null,null,null,null,7.582,38.5294,7.3899,54.1715,58.3879,0.9648
internas,reg,LightGBM,2026-07-01,244689,null,null,null,null,null,6.376,25.1763,6.7707,41.2589,65.0249,0.9775
internas,reg,LightGBM,2026-08-01,135690,null,null,null,null,null,5.8449,18.4929,6.7305,37.1722,59.6505,0.9763


In [0]:
# Variables con más peso en validación (promedio de los bloques)
display(imp.groupby(["tarea", "variable"]).importancia.mean().sort_values(ascending=False).groupby(level=0).head(8).round(4))

tarea  variable            
reg    hist_h_zona_franja      0.3257
       categoria_zona          0.2048
       margen_sla_en_t0_h      0.1963
       dia_semana              0.1603
clf    margen_sla_en_t0_h      0.1054
reg    hist_inc_zona_franja    0.1001
       hist_h_cliente          0.0633
       ID_PRODUCTO             0.0558
clf    dia_semana              0.0533
reg    n_hist_cliente          0.0406
clf    hist_inc_zona_franja    0.0355
       categoria_zona          0.0347
       ID_PRODUCTO             0.0085
       hist_h_zona_franja      0.0072
       hist_inc_cliente        0.0060
       ID_TIPOGUIA             0.0042
Name: importancia, dtype: float64

## 5. Guardar para 0416

In [0]:
pred.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}.parquet", index=False)
imp.to_parquet(f"{CARPETA_MODELADO}importancia_{MODELO}.parquet", index=False)
if len(pred_perdida):
    pred_perdida.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}_por_perdida.parquet", index=False)
with open(RUTA_PARAMS, "w") as f:
    json.dump({"modelo": MODELO, "espacio": ESPACIO, "params_base": PARAMS_BASE, "semilla": SEMILLA,
               "n_configs": N_CONFIGS_BUSQUEDA, "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES,
               "mejores_por_perdida": MEJORES_POR_PERDIDA}, f, indent=2, default=str)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_11/V4/MODELADO/
